In [ ]:
!pip install nltk
!pip install rapidfuzz
!pip install beautifulsoup4
!pip install requests
!pip install flask
!pip install fastapi
%pip install rapidfuzz


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ ------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 2.8 MB/s eta 0:00:01
   ---------- ----------------------------- 0.5/2.0 MB 2.8 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 1.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.0 MB/s eta 0:00:00

   ---------- ----------------------------- 2/8 [anyio]
   ---------- ----------------------------- 2/8 [anyio]
   ---------- ----------------------------- 2/8 [anyio]
   --------------- ------------------------ 3/8 [annotated-types]
   ------------------------- -------------- 5/8 [starlette]
   ------------------------- -------------- 5/8 [starlette]
   ------------------------------ --------- 6/8 [pydantic]
   ------------------------------ --------- 6/8 [pydantic]
   ------------------------------ --------- 6/8 [pydantic]
   ----------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re
import string
from difflib import get_close_matches
import sys
import subprocess
import json
import os

In [77]:
from rapidfuzz import process, fuzz
# Correcteur orthographe


text = "mianatr izahay androan marain"
print('Teny orizinaly')
print(text)

# Etape3 : tokenisation et normalisation 
def tokenisation_regex(text):
    text = re.sub(r"(\w+n)'ny\b", r"\1 ny", text, flags=re.IGNORECASE)
    text = re.sub(r"\bamin'ny\b", "amin ny", text, flags=re.IGNORECASE)
    text = re.sub(r"\bh'(\w+)", r"ho \1", text, flags=re.IGNORECASE)
    text = re.sub(r"\ban'(\w+)", r"any \1", text, flags=re.IGNORECASE)

    pattern = r"\w+|[^\w\s]"
    return re.findall(pattern, text, re.UNICODE)

token = tokenisation_regex(text)
print(f"tokenisation ({len(token)}tokens) : ")
print(token)

Teny orizinaly
mianatr izahay androan marain
tokenisation (4tokens) : 
['mianatr', 'izahay', 'androan', 'marain']


In [78]:
# Etape 4: chargement de dictionnaire
with open('malagasy_roots_final.json', 'r', encoding='utf-8') as f:
    dico_data = json.load(f)

tous_les_mots = []

mapping_racines = {}

for racine, derives in dico_data.items():
    racine_clean = racine.lower()
    tous_les_mots.append(racine_clean)
    mapping_racines[racine_clean] = racine_clean
    
    for mot in derives:
        mot_clean = mot.lower()
        tous_les_mots.append(mot_clean)
        mapping_racines[mot_clean] = racine_clean

mots_valides = set(tous_les_mots)

print(f"✅ Dictionnaire chargé avec succès.")
print(f"Nombre de racines : {len(dico_data)}")
print(f"Nombre total de formes reconnues : {len(mots_valides)}")

✅ Dictionnaire chargé avec succès.
Nombre de racines : 5673
Nombre total de formes reconnues : 45834


In [79]:
# Etape5: detection des erreurs
def detecteur_erreur_malagasy(text, reference_vocab):
    mots = re.findall(r"\b\w+(?:'\w+)?\b", text.lower(), re.UNICODE)
    erreurs = []
    for mot in mots:
        if mot not in reference_vocab:

            suggestions = get_close_matches(mot, list(reference_vocab), n=1, cutoff=0.6)
            suggestion = suggestions[0] if suggestions else None
            erreurs.append({
                'mot_errone': mot,
                'suggestion_corrigee': suggestion if suggestion else "Aucune suggestion",

                'index_dans_texte': text.lower().find(mot)
            })
            
    return erreurs

resultats = detecteur_erreur_malagasy(text, mots_valides)

if not resultats:
    print(" Aucune erreur détectée. Tous les mots sont dans le dictionnaire.")
else:
    print(f" {len(resultats)} erreur(s) trouvée(s) :")
    df_erreurs = pd.DataFrame(resultats)
    display(df_erreurs)


 3 erreur(s) trouvée(s) :


,mot_errone,suggestion_corrigee,index_dans_texte
0,mianatr,mianatra,0
1,androan,androany,15
2,marain,maraina,23


In [80]:
#Etape 6 : Recherche de candidats, calcul de similarité selection du meilleur candidat

def recherche_candidat_rapidfuzz(text, reference_vocab):
    mots_texte = re.findall(r"\b\w+(?:'\w+)?\b", text.lower(), re.UNICODE)
    resultats_correction = []
  
    vocab_list = list(reference_vocab)

    for mot in mots_texte:

        if mot not in reference_vocab:
    
            match = process.extractOne(mot, vocab_list, scorer=fuzz.ratio)
            
            if match:
                candidat, score, index = match
            
                if score >= 75:
                    resultats_correction.append({
                        'mot_tape': mot,
                        'correction_proposee': candidat,
                        'score_ressemblance': f"{round(score, 1)}%"
                    })
                else:
                    resultats_correction.append({
                        'mot_tape': mot,
                        'correction_proposee': "Aucun match proche",
                        'score_ressemblance': f"{round(score, 1)}%"
                    })
                
    return resultats_correction
matchings = recherche_candidat_rapidfuzz(text, mots_valides)
df_matches = pd.DataFrame(matchings)
print("Résultats du Matching RapidFuzz :")
display(df_matches)


Résultats du Matching RapidFuzz :


,mot_tape,correction_proposee,score_ressemblance
0,mianatr,mianatra,93.3%
1,androan,androana,93.3%
2,marain,maraina,92.3%


In [82]:
#Etape 7 : validation, correction et reconstruction
def validation_et_correction_ortho(text, reference_vocab, seuil=80):
    # 1. Découpage en mots et ponctuation
    mots_bruts = re.findall(r"\b\w+(?:'\w+)?\b|\S", text, re.UNICODE)
    texte_corrige = []
    journal_corrections = []
    vocab_list = list(reference_vocab)
    for mot in mots_bruts:
    
        if re.match(r"\w+", mot):
            mot_lower = mot.lower()
            if mot_lower in reference_vocab:
                texte_corrige.append(mot)
            else:
              
                match = process.extractOne(mot_lower, vocab_list, scorer=fuzz.ratio)
                
                if match:
                    candidat, score, _ = match
                    if score >= seuil:

                        correction = candidat.capitalize() if mot[0].isupper() else candidat
                        texte_corrige.append(correction)
                        journal_corrections.append((mot, correction, f"{score:.1f}%"))
                    else:
                        
                        texte_corrige.append(mot)
                else:
                    texte_corrige.append(mot)
        else:
           
            texte_corrige.append(mot)


    phrase_finale = " ".join(texte_corrige)
    phrase_finale = re.sub(r'\s+([.,!?;:])', r'\1', phrase_finale) # mot . -> mot.
    phrase_finale = re.sub(r"(\w+)\s*'\s*(ny|ala|izany)", r"\1'\2", phrase_finale) # amin ' ny -> amin'ny
    
    return phrase_finale.strip(), journal_corrections


text_exemple = "Mianatr anaty sekol izahi." 

text_propre, logs = validation_et_correction_ortho(text, mots_valides, seuil=80)

print(f"Texte Original : {text_exemple}")
print(f"Texte Corrigé  : {text_propre}")
print("\nJournal des décisions :")
if not logs:
    print("Aucune modification effectuée.")
else:
    for erreur, correction, score in logs:
        print(f"REMPLACÉ : '{erreur}' -> '{correction}' (Confiance: {score})")

Texte Original : Mianatr anaty sekol izahi.
Texte Corrigé  : mianatra izahay androana maraina

Journal des décisions :
REMPLACÉ : 'mianatr' -> 'mianatra' (Confiance: 93.3%)
REMPLACÉ : 'androan' -> 'androana' (Confiance: 93.3%)
REMPLACÉ : 'marain' -> 'maraina' (Confiance: 92.3%)
